In [1]:
!pip uninstall -q -y dataproc-spark-connect
!pip install -q pyspark==3.5.1
!pip install -q findspark

import findspark
findspark.init()
from pyspark.sql import SparkSession

# Initialize Spark Session
spark = SparkSession.builder.appName("CSV_Read").getOrCreate()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.9 MB/s eta 0:00:00


In [2]:
df = spark.read.csv(f'/content/drive/MyDrive/Dados Projeto/empreendimento-geracao-distribuida.csv', header=True, inferSchema=True, sep=';')
#df.printSchema()

In [ ]:
# Total 4624393
df.count()

4624393

In [ ]:
df.describe().show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.13/dist-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/lib/python3.13/socket.py", line 723, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Total de distribuidoras
df.select('SigAgente').distinct().count()

In [ ]:
# Exibindo as distribuidoras
df.select('SigAgente').distinct().show(103)

In [3]:
# Aplicando filtro da Neoenergia PE
df_filtrado = df.filter(df.SigAgente == 'Neoenergia PE')

In [ ]:
# Total da Neoenergia PE 172993
df_filtrado.count()

172993

In [ ]:
# Exibindo os estados vinculados a Neoenergia PE
df_filtrado.select('SigAgente','SigUF').groupBy('SigUF').count().show()

In [4]:
# Excluindo estados diferentes de PE
df_filtrado = df_filtrado.filter(df_filtrado.SigUF == 'PE')

In [ ]:
# Total após 2 limpeza, que removeu estados de fora de PE 172080
df_filtrado.count()

172080

In [ ]:
df_filtrado.select('SigTipoGeracao','SigTipoGeracao').groupBy('SigTipoGeracao').count().show()

In [5]:
df_filtrado = df_filtrado.filter(df_filtrado.SigTipoGeracao == 'UFV')

In [ ]:
# Total após 3 limpeza, que removeu outros tipos de geração diferentes de UFV 172014
df_filtrado.count()

172014

In [6]:
# Listas das entidades que precisamos
titular = ['NumCPFCNPJ','NomTitularEmpreendimento'] #ok
distribuidora = ['NumCNPJDistribuidora','SigAgente','NomAgente'] #ok
municipios = ['NomMunicipio','SigUF','NomRegiao'] #ok
empreendimento = ['CodEmpreendimento','DscClasseConsumo','DscSubGrupoTarifario','MdaPotenciaInstaladaKW','AnmPeriodoReferencia','DscPorte','CodCEP','QtdUCRecebeCredito','DthAtualizaCadastralEmpreend','DscModalidadeHabilitado'] #ok

# Checagem de valores nulos nas colunas das entidades

In [ ]:
# Checagem de valores nulos nas colunas
print('Valores nulos em titulares:')
for col_name in titular:
    null_count = df_filtrado.filter(df_filtrado[col_name].isNull()).count()
    print(f"  Coluna '{col_name}': {null_count} Valores nulos")

In [7]:
df_filtrado = df_filtrado.na.drop(subset=titular)

In [8]:
# Total após 4 limpeza, que removeu as linas com valores nulos de titular 172009
df_filtrado.count()

172009

In [ ]:
print('Valores nulos em distribuidora:')
for col_name in distribuidora:
    null_count = df_filtrado.filter(df_filtrado[col_name].isNull()).count()
    print(f"  Coluna '{col_name}': {null_count} Valores nulos")

In [ ]:
print('Valores nulos em municipios:')
for col_name in municipios:
    null_count = df_filtrado.filter(df_filtrado[col_name].isNull()).count()
    print(f"  Coluna '{col_name}': {null_count} Valores nulos")

In [ ]:
print('Valores nulos em empreendimento:')
for col_name in empreendimento:
    null_count = df_filtrado.filter(df_filtrado[col_name].isNull()).count()
    print(f"  Coluna '{col_name}': {null_count} Valores nulos")

Valores nulos em empreendimento:
  Coluna 'CodEmpreendimento': 0 Valores nulos
  Coluna 'DscClasseConsumo': 0 Valores nulos
  Coluna 'DscSubGrupoTarifario': 0 Valores nulos
  Coluna 'MdaPotenciaInstaladaKW': 0 Valores nulos
  Coluna 'AnmPeriodoReferencia': 0 Valores nulos
  Coluna 'DscPorte': 0 Valores nulos
  Coluna 'CodCEP': 0 Valores nulos
  Coluna 'QtdUCRecebeCredito': 0 Valores nulos
  Coluna 'DthAtualizaCadastralEmpreend': 0 Valores nulos
  Coluna 'DscModalidadeHabilitado': 0 Valores nulos


In [11]:
# Linhas atuais no df_filtrado 172009, linhas de CPF não duplicados 96198
df_filtrado.select('NumCPFCNPJ',).distinct().count()

96198